In [5]:
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

load_dotenv()

pg_host = os.getenv("PG_HOST")
pg_port = os.getenv("PG_PORT")
pg_db = os.getenv("PG_DATABASE")
pg_user = os.getenv("PG_USER")
pg_password = os.getenv("PG_PASSWORD")

print("Host OK:", pg_host is not None)
print("DB OK:", pg_db is not None)
print("User OK:", pg_user is not None)
print("Password OK:", pg_password is not None)

connection_string = f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_db}"
engine = create_engine(connection_string)

with engine.connect() as conn:
    resultado = conn.execute(text("SELECT version();"))
    print(resultado.fetchone())

Host OK: True
DB OK: True
User OK: True
Password OK: True
('PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit',)


In [6]:
import pandas as pd
from sqlalchemy import text

df_final = pd.read_csv("../data/processed/inflacion_procesada.csv", parse_dates=["mes"])

print(df_final.shape)
df_final.head()

(132, 5)


,mes,ipc_valor,uf_valor,indice_ipc,indice_uf
0,2015-01-01,68.135885,24557.15,100.000000,100.000000
1,2015-02-01,68.375560,24545.23,100.351759,99.951460
2,2015-03-01,68.805504,24622.78,100.982770,100.267254
3,2015-04-01,69.201029,24754.77,101.563264,100.804735
4,2015-05-01,69.323289,24904.75,101.742700,101.415474


In [7]:
upsert_query = text("""
    INSERT INTO inflacion_mensual (mes, ipc_valor, uf_valor, indice_ipc, indice_uf)
    VALUES (:mes, :ipc_valor, :uf_valor, :indice_ipc, :indice_uf)
    ON CONFLICT (mes)
    DO UPDATE SET
        ipc_valor = EXCLUDED.ipc_valor,
        uf_valor = EXCLUDED.uf_valor,
        indice_ipc = EXCLUDED.indice_ipc,
        indice_uf = EXCLUDED.indice_uf,
        fecha_actualizacion = NOW()
""")

# Convertimos el DataFrame a una lista de diccionarios — una fila = un diccionario
registros = df_final.to_dict(orient="records")

with engine.connect() as conn:
    conn.execute(upsert_query, registros)
    conn.commit()   # sin esto, PostgreSQL guarda los cambios en un borrador temporal y los descarta

print(f"Cargados/actualizados {len(registros)} registros.")

Cargados/actualizados 132 registros.
